
# 01 - Ingestão Bronze

Os arquivos CSV são carregados para a camada Bronze.

A ideia é simples: a Bronze funciona como um **depósito organizado**.

Os dados são armazenados praticamente como chegaram, mas recebem algumas informações de rastreabilidade, como:

- arquivo de origem
- data e hora da ingestão

Limpeza, correções e regras de negócio serão aplicadas somente na camada Silver.

In [0]:
#Define onde estão os arquivos originais e onde as tabelas Bronze serão gravadas
from pyspark.sql import functions as F

CATALOG = "nautical_lighthouse"
BRONZE_SCHEMA = "bronze"
RAW_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/raw_files"
print(f"Origem: {RAW_PATH}")
print(f"Destino: {CATALOG}.{BRONZE_SCHEMA}")

In [0]:
#Verificação inicial para saber quais arquivos existem

csv_files = [
    file for file in dbutils.fs.ls(RAW_PATH) if file.name.lower().endswith(".csv")
]
print(f"Arquivos CSV: {len(csv_files)}")

for file in sorted(csv_files, key=lambda x: x.name):
    print(file.name)

In [0]:
#Testando a leitura de um arquivo

sample_path = f"{RAW_PATH}/orders.csv"
df_orders_sample = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(sample_path)

)
display(df_orders_sample.limit(10))

In [0]:
#Verificar como o Spark interpretou os tipos de cada coluna

df_orders_sample.printSchema()

In [0]:
#Conferência do volume
orders_count = df_orders_sample.count()
print(f"Registros em orders.csv {orders_count:,}")

In [0]:
#Preparação da ingestão: criar uma rotina única para carregar todos os arquivos.

from pyspark.sql import functions as F

In [0]:
#Função de ingestão: Única rotina para carregar todos os arquivos

def ingest_to_bronze(file):
    file_name = file.name
    table_name = file_name.replace(".csv", "")

    file_path = f"{RAW_PATH}/{file_name}"
    table_full_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
        .withColumn("source_file", F.lit(file_name))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_full_name)
    )

    return table_name, df.count()

In [0]:
#Teste de ingestão com 'orders.csv'

orders_file = next(
    file for file in csv_files
    if file.name == "orders.csv"
)

table_name, row_count = ingest_to_bronze(orders_file)

print(f"{table_name}: {row_count:,} registros")

In [0]:
#Validação do teste

df_bronze_orders = spark.table(
    "nautical_lighthouse.bronze.orders"
)

print(f"Registros na Bronze: {df_bronze_orders.count():,}")

display(df_bronze_orders.limit(10))

In [0]:
#Carga da camada Bronze

ingestion_results = []

for file in sorted(csv_files, key=lambda x: x.name):
    table_name, row_count = ingest_to_bronze(file)

    ingestion_results.append(
        (table_name, row_count)
    )

    print(f"{table_name}: {row_count:,} registros")

In [0]:
#Conferência das tabelas

bronze_tables = spark.sql(
    f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}"
)

display(bronze_tables)

In [0]:
#Validação da quantidade: 24 arquivos vs 24 tabelas Bronze

bronze_table_count = bronze_tables.count()

print(f"Tabelas Bronze encontradas: {bronze_table_count}")





In [0]:
#Resumo da ingestão

ingestion_summary = spark.createDataFrame(
    ingestion_results,
    ["table_name", "row_count"]
)

display(
    ingestion_summary.orderBy("table_name")
)

In [0]:
%sql
SELECT
    COUNT(*) AS total_linhas,
    MIN(created_at) AS data_minima,
    MAX(created_at) AS data_maxima,
    ROUND(MIN(total), 2) AS valor_minimo,
    ROUND(MAX(total), 2) AS valor_maximo,
    ROUND(AVG(total), 2) AS valor_medio
FROM nautical_lighthouse.bronze.orders;

In [0]:
%sql
SELECT
    COUNT(*) AS total_colunas
FROM nautical_lighthouse.information_schema.columns
WHERE table_schema = 'bronze'
  AND table_name = 'orders'
  AND column_name NOT IN ('source_file', 'ingestion_timestamp');

### Resultado

A camada Bronze agora contém uma cópia estruturada dos 24 arquivos recebidos.

Os dados ainda não foram corrigidos ou transformados. Essa etapa será feita na camada Silver, onde vamos tratar qualidade, tipos, duplicidades e regras de negócio.